# Discern: Open-Set Person Re-Identification Under Low Appearance Variance
## Full T4 GPU Ablation Training Suite on Market-1501 (30+ Epochs, All Layers Unfrozen)

This ready-to-run notebook fine-tunes **OSNet x0.5** across 4 distinct ablation models on a single NVIDIA T4 GPU:
1. **Model 1 (Baseline)**: OSNet x0.5 + CrossEntropy Loss + Uniform Random PK
2. **Model 2 (+ Margin Loss)**: OSNet x0.5 + ArcFace Margin Loss + Uniform Random PK
3. **Model 3 (+ Look-Alike Mining)**: OSNet x0.5 + ArcFace Loss + Look-Alike Cluster PK Mining
4. **Model 4 (+ Horizontal Stripes)**: OSNet x0.5 with 3 Stripe Heads + ArcFace Loss + Look-Alike Mining

After training, it benchmarks standard Market-1501 CMC (Rank-1, Rank-5, mAP) and held-out open-set metrics (AUROC, TAR@1%FAR, DIR@1%FAR with 95% bootstrap CIs).

In [ ]:
# 1. Environment Verification & GPU Check
!nvidia-smi
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device   : {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone or Link Codebase & Install Dependencies
!pip install -q gdown scikit-learn matplotlib torchvision
import os, sys
if not os.path.exists('discern'):
    # If running directly inside the cloned repo:
    if os.path.exists('backend'):
        PROJECT_ROOT = os.getcwd()
    else:
        !git clone https://github.com/zed-digital/discern.git
        PROJECT_ROOT = os.path.join(os.getcwd(), 'discern')
        os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = os.path.join(os.getcwd(), 'discern')
    os.chdir(PROJECT_ROOT)

sys.path.insert(0, os.getcwd())
print(f"Working Directory: {os.getcwd()}")

In [ ]:
# 3. Download Official Pretrained OSNet x0.5 Weights
os.makedirs('weights', exist_ok=True)
weights_path = 'weights/osnet_x0_5_msmt17.pth'
if not os.path.exists(weights_path):
    print("Downloading official Kaiyang Zhou OSNet x0.5 MSMT17 weights...")
    !gdown "16DGLbZukvVYgINws8u8deSaOqjybZ83i" -O {weights_path}
print(f"Weights status: {os.path.exists(weights_path)} ({os.path.getsize(weights_path) / 1024 / 1024:.2f} MB)")

In [ ]:
# 4. Prepare Dataset & Open-Set Protocol Split
!python scripts/prepare_data.py
!python scripts/cluster_uniforms.py

In [ ]:
# 5. Run Zero-Shot Pretrained Backbone Sanity Check
!python scripts/test_raw_osnet.py

In [ ]:
# 6. Train All 4 Ablation Checkpoints (30 Epochs, Unfrozen Backbone, T4 GPU)
!python scripts/kaggle_train.py \
    --device cuda \
    --epochs 30 \
    --batch-p 8 \
    --batch-k 4 \
    --lr 0.0003 \
    --unfreeze-all

In [ ]:
# 7. Run Scientific Evaluation Suite with 95% Bootstrap Confidence Intervals
!python scripts/evaluate.py

In [ ]:
# 8. Display Final Verified Table & Visualizations
import json
import pandas as pd
from IPython.display import Image, display

with open('results/evaluation_results.json', 'r') as f:
    res = json.load(f)

rows = []
for r in res.get('ablation_study', []):
    rows.append({
        'Configuration': r['component'],
        'Full AUROC': f"{r['full_auroc']:.4f} [{r['full_auroc_ci'][0]:.3f}, {r['full_auroc_ci'][1]:.3f}]",
        'Full TAR@1%': f"{r['full_tar_1pct']*100:.1f}% [{r['full_tar_1pct_ci'][0]*100:.1f}%, {r['full_tar_1pct_ci'][1]*100:.1f}%]",
        'LowVar TAR@1%': f"{r['lowvar_tar_1pct']*100:.1f}% [{r['lowvar_tar_1pct_ci'][0]*100:.1f}%, {r['lowvar_tar_1pct_ci'][1]*100:.1f}%]",
    })

df = pd.DataFrame(rows)
display(df)

if os.path.exists('results/roc_chart.png'):
    display(Image('results/roc_chart.png'))